**Goal** : This notebook extracts information from `dpcstruct(_classification.tsv,mcs_reps.fasta)` and outputs a consistent dataframe whose columns summarize mcs domains(proteins).

In [1]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport

# I. dpcstruct_classification.tsv file

In [2]:
# dpcstruct_classification.tsv data
path_to_dpcstruct_classification = "/u/mdmc/enyanduk/internship_areasciencepark/Data/dpcstruct/dpcstruct_classification.tsv"

In [3]:
with open(path_to_dpcstruct_classification, "r",encoding="utf-8") as f:
  print(repr(f.readline()))

'mcID proteinID start end\n'


In [4]:
# It's a space-separated file, let's explore it:
df_dpcstruct_class = pd.read_csv(path_to_dpcstruct_classification, sep=r"\s+")
df_dpcstruct_class.head()

,mcID,proteinID,start,end
0,0,A0A1Q3ZAL7,4,118
1,0,A0A3P1XLE5,3,122
2,0,A0A2V5NFM0,9,100
3,0,A0A6P1ZDY0,24,124
4,0,A0A6P1ZFS3,1,87


In [5]:
# Info about the dataset
df_dpcstruct_class.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1642061 entries, 0 to 1642060
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   mcID       1642061 non-null  int64 
 1   proteinID  1642061 non-null  object
 2   start      1642061 non-null  int64 
 3   end        1642061 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 50.1+ MB


In [6]:
# We perfom some transformation to the dataset:
df1 = df_dpcstruct_class.copy()
# T1. We  join start and end by "-"
df1["prot_range"] = df1["start"].astype(str) + "-" + df1["end"].astype(str)
# T2 . We drop the start and end columns
df1.drop(columns=["start", "end"], inplace=True)
# T3. We rename columns properly
df1.rename(columns={"mcID": "mc_id","proteinID":"protein_id"}, inplace=True)
# T4. mc_id stores integer values, we convert them to integer type
df1["mc_id"] = df1["mc_id"].astype(int)
# T5 : We sort the data by mc_id in ascending order
df1.sort_values(by="mc_id",ascending=True,inplace=True)
# T6 : Rewrite each ID in mc_id column as MCID: e.g.: 1 -> MC1
df1["mc_id"] = df1["mc_id"].apply(lambda x: f"MC{x}")
# T7 : We reset the index
df1.reset_index(drop=True, inplace=True)
df1.head()

,mc_id,protein_id,prot_range
0,MC0,A0A1Q3ZAL7,4-118
1,MC0,A0A170YWQ7,2-107
2,MC0,A0A3E2NVG7,1-133
3,MC0,A0A2W4V3S2,3-113
4,MC0,A0A537IMV5,3-112


# II. Parse representative sequences (`mcs_reps.fasta` file)

In [7]:
# Parse the mcs_reps.fasta file
path_to_fasta = "/u/mdmc/enyanduk/internship_areasciencepark/Data/dpcstruct/mcs_reps.fasta"

records = []
with open(path_to_fasta, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            # Parse header: e.g. ">A0A009K0I0_6579 | 61-148"
            header = line[1:]  # remove ">"
            parts = header.split(" | ")
            representative = parts[0].strip()          # e.g. "A0A009K0I0_6579"
            prot_range = parts[1].strip()               # e.g. "61-148"
            # Split representative into protein_id and mc_id
            last_underscore = representative.rfind("_")
            protein_id = representative[:last_underscore]   # e.g. "A0A009K0I0"
            mc_num = representative[last_underscore + 1:]   # e.g. "6579"
            mc_id = mc_num                           # e.g. "6579"
        else:
            # Sequence line
            prot_seq = line
            records.append({
                "representative": representative,
                "protein_id": protein_id,
                "mc_id": mc_id,
                "prot_range": prot_range,
                "prot_seq": prot_seq
            })

df_mcs_reps = pd.DataFrame(records)
print(f"Shape: {df_mcs_reps.shape}")
df_mcs_reps.head()

Shape: (56438, 5)


,representative,protein_id,mc_id,prot_range,prot_seq
0,A0A009IR62_1789,A0A009IR62,1789,1-186,MSLMKSVVLIFASLAVNIAYSAETNPSIQNYWSIAEQKKLDQDITW...
1,A0A009JJQ9_20571,A0A009JJQ9,20571,7-90,IPLIDKLVKEKNFVLLTWDARYSSGAWACCLPYLNQCEVVYEASED...
2,A0A009K0I0_6579,A0A009K0I0,6579,61-148,MDKQFKQGLKKASAENSPFSALGYILTTGANWAKPIENFKLTIERD...
3,A0A009QKA3_27491,A0A009QKA3,27491,1-105,MKPEQFIREHGEKKARALLAQLHNLGCPDDMKITVINGMWHRTSKG...
4,A0A010R0N5_15978,A0A010R0N5,15978,9-507,LTAEQTKALFNILTHFETYHEIEDFKKPETISDYGYPFAQSSKPGE...


In [8]:
# Some transformations to the dataframe:
df2 = df_mcs_reps.copy()
# mc_id is currently a string, we convert it to int for sorting
df2["mc_id"] = df2["mc_id"].astype(int)
# T1 : We sort the data by mc_id in ascending order
df2.sort_values(by="mc_id",ascending=True,inplace=True)
# T2 : We reset the index
df2.reset_index(drop=True, inplace=True)
# T3 : Rewrite each ID in mc_id column as MCID: e.g.: 1 -> MC1
df2["mc_id"] = df2["mc_id"].apply(lambda x: f"MC{x}")
# Head of the transformed dataframe
df2.head(10)

,representative,protein_id,mc_id,prot_range,prot_seq
0,A0A1Q3ZAL7_0,A0A1Q3ZAL7,MC0,4-118,LIFLFVLTTTSFTTNLFKAQVNVNININSQPEWGPRGYNYVESYYL...
1,A0A537IMV5_0,A0A537IMV5,MC0,3-112,KIFICFALCLLAAFFKPASAQFSTNENIKDQPKWGLAGQKYVEYYY...
2,A0A7Y5H9I3_1,A0A7Y5H9I3,MC1,1-65,MKEVVSVAVLASKEGFNVYDLADGHTIKMKTVVLDVVRVEGVKDEL...
3,A0A497KZ75_1,A0A497KZ75,MC1,9-72,SVDLDFTEVEEHWNCYKLSDGTTLKVKLVLRGVKRLNRYEPDGTPI...
4,A0A7Y5HAR7_2,A0A7Y5HAR7,MC2,518-906,GLSKQEALRSEVFGLIARVRQMGVPADGFVYYERGWYTGTEFTYAK...
5,A0A2A2QTX3_2,A0A2A2QTX3,MC2,20-353,NTPPPEFVERAGQWMISTDPSKRQAAYRSWLQLGPGTMEAYEKALR...
6,A0A2E0QQW9_3,A0A2E0QQW9,MC3,20-394,LPFNKCQQEFSAIGLNSIVPGHGDPIAKWLVGGGVGRRILQIFIVL...
7,A0A849WCM9_3,A0A849WCM9,MC3,318-702,DLREEIAWMHNDPIGMFFEKSRLSIFSVFAFFFILDAIVVVVWALI...
8,A0A372JJE4_4,A0A372JJE4,MC4,24-156,CSVQIRVGKLLLTSLNVTQTYGGLIEGGPSKRFNDRLVQRALESAS...
9,A0A849WD35_4,A0A849WD35,MC4,3-135,FIELKCGRSVYISDFDYTRTYGGLIEGMPNEKLNNKIIECAMIRMQ...


In [9]:
# Info
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56438 entries, 0 to 56437
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   representative  56438 non-null  object
 1   protein_id      56438 non-null  object
 2   mc_id           56438 non-null  object
 3   prot_range      56438 non-null  object
 4   prot_seq        56438 non-null  object
dtypes: object(5)
memory usage: 2.2+ MB


# III. Create per-MC FASTA files (`dpcstruct_reps_seqs/`)
For each unique `mc_id`, create a file `MCID.fasta` containing all representative sequences belonging to that metacluster, in the original FASTA format:
```
>Protein_ID | Range
Sequence
```

In [10]:
output_dir = "/u/mdmc/enyanduk/internship_areasciencepark/Data/dpcstruct/dpcstruct_reps_seqs"
os.makedirs(output_dir, exist_ok=True)

# Group by mc_id and write one FASTA file per metacluster
for mc_id, group in df2.groupby("mc_id"):
    fasta_path = os.path.join(output_dir, f"{mc_id}.fasta")
    with open(fasta_path, "w", encoding="utf-8") as f:
        for _, row in group.iterrows():
            f.write(f">{row['representative']} | {row['prot_range']}\n")
            f.write(f"{row['prot_seq']}\n")

print(f"Created {df2['mc_id'].nunique()} FASTA files in {output_dir}")

Created 28246 FASTA files in /u/mdmc/enyanduk/internship_areasciencepark/Data/dpcstruct/dpcstruct_reps_seqs


In [11]:
# Verify that df2 is a subset of df1 on (protein_id, mc_id, prot_range)
join_cols = ["protein_id", "mc_id", "prot_range"]

# Create sets of tuples for comparison
df2_keys = set(df2[join_cols].itertuples(index=False, name=None))
df1_keys = set(df1[join_cols].itertuples(index=False, name=None))

missing = df2_keys - df1_keys
is_subset = len(missing) == 0

print(f"df2 rows: {len(df2_keys)}")
print(f"df1 rows: {len(df1_keys)}")
print(f"df2 rows NOT in df1: {len(missing)}")
print(f"Is df2 a subset of df1? {is_subset}")

if not is_subset:
    print("\nSample missing rows:")
    for row in list(missing)[:5]:
        print(row)

df2 rows: 56438
df1 rows: 1642061
df2 rows NOT in df1: 0
Is df2 a subset of df1? True


In [12]:
# Create df3 by merging df1 with df2's prot_seq column
# Left join keeps all df1 rows; rows without a representative sequence get NaN in prot_seq
df3 = df1.merge(df2[["protein_id", "mc_id", "prot_range", "prot_seq"]],
                on=["protein_id", "mc_id", "prot_range"],
                how="left")

# Reorder columns: mc_id, protein_id, prot_range, prot_seq
df3 = df3[["mc_id", "protein_id", "prot_range", "prot_seq"]]

print(f"df3 shape: {df3.shape}")
print(f"Rows with sequence: {df3['prot_seq'].notna().sum()}")
print(f"Rows without sequence: {df3['prot_seq'].isna().sum()}")
df3.head(10)

df3 shape: (1642061, 4)
Rows with sequence: 56438
Rows without sequence: 1585623


,mc_id,protein_id,prot_range,prot_seq
0,MC0,A0A1Q3ZAL7,4-118,LIFLFVLTTTSFTTNLFKAQVNVNININSQPEWGPRGYNYVESYYL...
1,MC0,A0A170YWQ7,2-107,NaN
2,MC0,A0A3E2NVG7,1-133,NaN
3,MC0,A0A2W4V3S2,3-113,NaN
4,MC0,A0A537IMV5,3-112,KIFICFALCLLAAFFKPASAQFSTNENIKDQPKWGLAGQKYVEYYY...
5,MC0,A0A4R2EMX1,49-158,NaN
6,MC0,A0A4R2EPD2,219-328,NaN
7,MC0,A0A533BJV2,138-258,NaN
8,MC0,A0A4Q3BL62,5-101,NaN
9,MC0,A0A161LV53,39-155,NaN


In [13]:
# Sort df3 so that within each mc_id, rows WITH a sequence come first, then NaN
df3["_has_seq"] = df3["prot_seq"].notna().astype(int)  # 1 = has seq, 0 = NaN
df3.sort_values(by=["mc_id", "_has_seq"], ascending=[True, False], inplace=True)
df3.drop(columns="_has_seq", inplace=True)
df3.reset_index(drop=True, inplace=True)
df3.head(10)

,mc_id,protein_id,prot_range,prot_seq
0,MC0,A0A1Q3ZAL7,4-118,LIFLFVLTTTSFTTNLFKAQVNVNININSQPEWGPRGYNYVESYYL...
1,MC0,A0A537IMV5,3-112,KIFICFALCLLAAFFKPASAQFSTNENIKDQPKWGLAGQKYVEYYY...
2,MC0,A0A170YWQ7,2-107,NaN
3,MC0,A0A3E2NVG7,1-133,NaN
4,MC0,A0A2W4V3S2,3-113,NaN
5,MC0,A0A4R2EMX1,49-158,NaN
6,MC0,A0A4R2EPD2,219-328,NaN
7,MC0,A0A533BJV2,138-258,NaN
8,MC0,A0A4Q3BL62,5-101,NaN
9,MC0,A0A161LV53,39-155,NaN


In [14]:
# info about df3
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1642061 entries, 0 to 1642060
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   mc_id       1642061 non-null  object
 1   protein_id  1642061 non-null  object
 2   prot_range  1642061 non-null  object
 3   prot_seq    56438 non-null    object
dtypes: object(4)
memory usage: 50.1+ MB


In [15]:
# Save to CSV
save_path = "/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_mcs_sequences.csv"
df3.to_csv(save_path, index=False)
print(f"Saved df3 ({df3.shape}) to {save_path}")

Saved df3 ((1642061, 4)) to /u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCStruct/dpcstruct_mcs_sequences.csv
